In [1]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt


# ============================================================
# USER SETTINGS
# ============================================================

ROOT = Path.cwd()

MAE_DATA_DIR = ROOT / "Data" / "testing_data"

# Full 2,000,000-sample test pool (replaces the old 1,000-sample subsample
# in-place as of 2026-09-12). Computed directly from each PNN's trained
# params.msgpack against the full contiguous test split (last 2M rows of
# each n's 20M-row dataset). The pre-replacement 1,000-sample files are kept
# as mae_values_n{4,6,8}_1000sample_OLD.npy.bak; see mae_summary_n{4,6,8}.json
# for the updated stats/provenance.
MAE_FILES = {
    4: MAE_DATA_DIR / "mae_values_n4.npy",
    6: MAE_DATA_DIR / "mae_values_n6.npy",
    8: MAE_DATA_DIR / "mae_values_n8.npy",
}

OUTPUT_DIR = ROOT / "Results" / "testing_mae_histograms"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [2]:

VERSION = 9  # v9: full 2,000,000-sample test pool (v8 was a 1,000-sample subsample)

SAVE_PNG = True
SAVE_PDF = True
SAVE_SVG = True
SAVE_DPI = 500


# ============================================================
# PLOT CONTROL PARAMETERS
# ============================================================

X_SCALE = 1e6
X_SCALE_LABEL = r"$\times 10^{-6}$"

NUM_BINS = 35

HIST_ALPHA = 0.85
HIST_COLOR = "skyblue"
HIST_EDGE_COLOR = "black"
HIST_LINEWIDTH = 1.8

MEAN_LINE_COLOR = "red"
MEAN_LINEWIDTH = 2.2

SHOW_GRID = False


# ============================================================
# FONT AND STYLE SETTINGS
# ============================================================

plt.rcParams.update({
    "font.size": 18,
    "axes.titlesize": 20,
    "axes.labelsize": 18,
    "xtick.labelsize": 16,
    "ytick.labelsize": 16,
    "legend.fontsize": 16,

    "lines.linewidth": 2.0,
    "axes.linewidth": 0.9,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3.5,
    "ytick.major.size": 3.5,

    "figure.dpi": 150,
    "savefig.dpi": SAVE_DPI,

    "figure.facecolor": "white",
    "axes.facecolor": "white",

    "pdf.fonttype": 42,
    "ps.fonttype": 42,

    "axes.unicode_minus": False,
})


# ============================================================
# FONT SIZE CONTROLS
# ============================================================

LABEL_FONTSIZE = 22
TICK_FONTSIZE = 18
MEAN_TEXT_FONTSIZE = 16

X_LABELPAD = 8
Y_LABELPAD = 8
TICK_PAD = 5


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def load_mae_values(path):
    """Load MAE values from .npy file."""
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"File not found:\n{path}")

    values = np.load(path)
    values = np.asarray(values, dtype=float).ravel()

    values = values[np.isfinite(values)]

    if len(values) == 0:
        raise ValueError(f"No valid MAE values found in:\n{path}")

    return values


def style_axis(ax):
    """Apply consistent paper-style axis formatting."""
    if SHOW_GRID:
        ax.grid(True, alpha=0.25, linewidth=0.6)
    else:
        ax.grid(False)

    ax.tick_params(
        axis="both",
        labelsize=TICK_FONTSIZE,
        width=0.8,
        length=3.5,
        direction="out",
        pad=TICK_PAD,
    )

    for spine in ax.spines.values():
        spine.set_linewidth(2.2)


def save_figure(fig, filename):
    """Save figure as PNG and/or PDF and/or SVG."""
    if SAVE_PNG:
        png_path = OUTPUT_DIR / f"{filename}.png"
        fig.savefig(png_path, dpi=SAVE_DPI, bbox_inches="tight")
        print(f"Saved PNG: {png_path.resolve()}")

    if SAVE_PDF:
        pdf_path = OUTPUT_DIR / f"{filename}.pdf"
        fig.savefig(pdf_path, bbox_inches="tight")
        print(f"Saved PDF: {pdf_path.resolve()}")

    if SAVE_SVG:
        svg_path = OUTPUT_DIR / f"{filename}.svg"
        fig.savefig(svg_path, bbox_inches="tight")
        print(f"Saved SVG: {svg_path.resolve()}")

    plt.close(fig)


## Separate per-node histograms

In addition to the combined 1x3 grid figure above, this section produces **3 separate
figures** (one per node count n=4, 6, 8), each with its own **"MAE distribution"**
legend entry (previously that label only appeared once, as a single legend shared
across the whole combined figure). Each figure also keeps its own "Mean = ..." line
entry, so both legend entries now appear together, per plot.

This reuses `load_mae_values`, `style_axis`, `save_figure`, and the existing style
constants (`MAE_FILES`, `NUM_BINS`, `X_SCALE`, `VERSION`, `OUTPUT_DIR`, etc.) defined
in the cell above -- no data regeneration needed.

In [3]:
def plot_single_histogram(n, mae_values):
    fig, ax = plt.subplots(figsize=(8, 7))

    mae_scaled = mae_values * X_SCALE
    mean_raw = np.mean(mae_values)
    mean_scaled = mean_raw * X_SCALE

    ax.hist(
        mae_scaled, bins=NUM_BINS, alpha=HIST_ALPHA,
        color=HIST_COLOR,
        edgecolor=HIST_EDGE_COLOR, linewidth=HIST_LINEWIDTH,
        label="MAE distribution",
    )

    ax.axvline(
        mean_scaled, color=MEAN_LINE_COLOR, linewidth=MEAN_LINEWIDTH,
        label=f"Mean = {mean_raw:.2e}",
    )

    # Both legend entries ("MAE distribution" + "Mean = ...") in every
    # individual plot, instead of the combined figure's single shared legend.
    ax.legend(
        loc="upper right", prop={"weight": "bold", "size": MEAN_TEXT_FONTSIZE}, frameon=True,
        framealpha=0.92, edgecolor="0.75", handlelength=1.8,
        borderpad=0.3, labelspacing=0.2,
    )

    ax.set_xlabel(f"Sample wise MAE ({X_SCALE_LABEL})", fontsize=LABEL_FONTSIZE, fontweight="bold", labelpad=X_LABELPAD)
    ax.set_ylabel("Sample Count", fontsize=LABEL_FONTSIZE, fontweight="bold", labelpad=Y_LABELPAD)

    style_axis(ax)
    fig.tight_layout()

    save_figure(fig, f"testing_mae_histogram_n{n}_v{VERSION}")
    plt.show()


for n, path in MAE_FILES.items():
    print(f"\nPlotting separate histogram for n = {n}")
    mae_values = load_mae_values(path)
    plot_single_histogram(n, mae_values)

print("\nAll 3 separate histograms generated.")


Plotting separate histogram for n = 4


Saved PNG: /home/bo48god/projects/deep_dreaming/PolySurrogate_temp/plotting_code/Results/testing_mae_histograms/testing_mae_histogram_n4_v9.png


Saved PDF: /home/bo48god/projects/deep_dreaming/PolySurrogate_temp/plotting_code/Results/testing_mae_histograms/testing_mae_histogram_n4_v9.pdf


Saved SVG: /home/bo48god/projects/deep_dreaming/PolySurrogate_temp/plotting_code/Results/testing_mae_histograms/testing_mae_histogram_n4_v9.svg

Plotting separate histogram for n = 6


Saved PNG: /home/bo48god/projects/deep_dreaming/PolySurrogate_temp/plotting_code/Results/testing_mae_histograms/testing_mae_histogram_n6_v9.png


Saved PDF: /home/bo48god/projects/deep_dreaming/PolySurrogate_temp/plotting_code/Results/testing_mae_histograms/testing_mae_histogram_n6_v9.pdf


Saved SVG: /home/bo48god/projects/deep_dreaming/PolySurrogate_temp/plotting_code/Results/testing_mae_histograms/testing_mae_histogram_n6_v9.svg

Plotting separate histogram for n = 8


Saved PNG: /home/bo48god/projects/deep_dreaming/PolySurrogate_temp/plotting_code/Results/testing_mae_histograms/testing_mae_histogram_n8_v9.png


Saved PDF: /home/bo48god/projects/deep_dreaming/PolySurrogate_temp/plotting_code/Results/testing_mae_histograms/testing_mae_histogram_n8_v9.pdf


Saved SVG: /home/bo48god/projects/deep_dreaming/PolySurrogate_temp/plotting_code/Results/testing_mae_histograms/testing_mae_histogram_n8_v9.svg

All 3 separate histograms generated.
